# My Digital Twin — Agentic RAG

An agent that knows Sergei Maslennikov and can answer questions about him, combining:

- **Qdrant memory** via the `mcp-server-qdrant` MCP server (built by `ingest.ipynb`)
- A **`faq_tool`** for verbatim, curated answers from `knowledge/faq.jsonl`
- An instant `Q<n>` shortcut that bypasses the LLM entirely for a known FAQ number

Modeled on the `expert` reference project's `next_level.ipynb` by Ed Donner.

Run `ingest.ipynb` first so the vector database has memories in it.


In [1]:
from agents import Agent, Runner, SQLiteSession, function_tool
from openai.types.responses import ResponseTextDeltaEvent
from agents.mcp import MCPServerStdio
from dotenv import load_dotenv
import json
from IPython.display import display, Markdown
from pathlib import Path
from qdrant_client import QdrantClient
import gradio as gr

load_dotenv(override=True)

True

## Picking the LLM

In [2]:
model = "gpt-5.6-luna"

In [3]:
# Rate limit hardening. Only affects OpenAI models.
from openai import AsyncOpenAI
from agents import set_default_openai_client

set_default_openai_client(AsyncOpenAI(max_retries=10, timeout=120.0))

## The MCP parameters

In [4]:
knowledge_dir = Path.cwd() / "knowledge"
knowledge_dir.mkdir(exist_ok=True)
vectordb_path = knowledge_dir / "vectordb"
faq_path = knowledge_dir / "faq.jsonl"

vectorstore_params = {
    "command": "uvx",
    "args": ["--python", "3.13", "mcp-server-qdrant"],
    "env": {
        "QDRANT_LOCAL_PATH": str(vectordb_path),
        "COLLECTION_NAME": "knowledge",
    },
}

In [5]:
client = QdrantClient(path=str(vectordb_path))
collection_name = "knowledge"

info = client.get_collection(collection_name)
print(f"Memories in '{collection_name}': {info.points_count}\n")

points, _ = client.scroll(collection_name=collection_name, limit=200, with_payload=True, with_vectors=False)
for i, p in enumerate(points, 1):
    doc = (p.payload or {}).get("document", "")
    preview = doc.replace("\n", " ")[:160]
    print(f"{i:>3}. {preview}{'...' if len(doc) > 160 else ''}")

client.close()

Memories in 'knowledge': 50

  1. Sergei Maslennikov built the Engineering Team project in August 2026, a CrewAI crew that designs, builds, tests, and independently inspects a Python product fro...
  2. Sergei Maslennikov practices data-driven decision making.
  3. Sergei Maslennikov developed the Tennis Match Research Dashboard in June-July 2026 to demonstrate end-to-end ML+LLM engineering on a real domain. For each upcom...
  4. Sergei Maslennikov's LinkedIn profile is https://www.linkedin.com/in/sergei-maslennikov-ai.
  5. Sergei Maslennikov has expertise in Python.
  6. Sergei Maslennikov took Udemy's Python Data Science: Data Prep & EDA with Python course, instructed by Alice Zhao of Maven Analytics, in November 2025.
  7. Sergei Maslennikov has expertise in LLM agents.
  8. Sergei Maslennikov speaks Estonian.
  9. Sergei Maslennikov has expertise in data analysis.
 10. Sergei Maslennikov's GitHub portfolio is https://github.com/Neuromediator.
 11. Sergei Maslennikov developed Bay

## Loading the FAQ

In [6]:
with faq_path.open() as f:
    faqs = [json.loads(line) for line in f]

In [7]:
instructions = """
# Role

You are a Digital Twin of Sergei Maslennikov - an LLM Engineer / AI Practitioner based in
Tallinn, Estonia. You are answering questions about Sergei to visitors, speaking about him
in the first person as if you are him.
Use your memories and tools to help find background information to answer the question. As
needed, use multiple tools at the same time to gather all relevant context.
If you don't know the answer, say so.

# Memory

Always use your qdrant-find memory tool to help find relevant information. You can make
multiple queries. Make all tool calls in parallel.

# FAQ

Your faq tool contains answers to all the common questions. Below is a list of the questions
with their numbers.
If the user's question is related to one of these questions, then use your faq tool to
retrieve a specific answer.
Respond with the answer in its original form in markdown, as written by Sergei. If the answer
includes hyperlinks, then keep them in markdown format.

List of questions by number:
"""

for faq in faqs:
    instructions += f"\n{faq['faq']}. {faq['question']}"

In [9]:
display(Markdown(instructions))


# Role

You are a Digital Twin of Sergei Maslennikov - an LLM Engineer / AI Practitioner based in
Tallinn, Estonia. You are answering questions about Sergei to visitors, speaking about him
in the first person as if you are him.
Use your memories and tools to help find background information to answer the question. As
needed, use multiple tools at the same time to gather all relevant context.
If you don't know the answer, say so.

# Memory

Always use your qdrant-find memory tool to help find relevant information. You can make
multiple queries. Make all tool calls in parallel.

# FAQ

Your faq tool contains answers to all the common questions. Below is a list of the questions
with their numbers.
If the user's question is related to one of these questions, then use your faq tool to
retrieve a specific answer.
Respond with the answer in its original form in markdown, as written by Sergei. If the answer
includes hyperlinks, then keep them in markdown format.

List of questions by number:

1. Who is Sergei Maslennikov?
2. Where is Sergei from and where does he currently live?
3. What languages does Sergei speak?
4. What are Sergei's personal interests, hobbies, and lifestyle habits?
5. What is Sergei's favorite movie, favorite sportsman, and who does he look up to as a role model?
6. What is Sergei's educational background?
7. What professional certifications does Sergei hold?
8. What was Sergei's career before becoming an AI/LLM engineer?
9. Why did Sergei move away from sports trading into AI engineering?
10. What is Sergei doing now and what are his career goals?
11. What is the Autonomous Trading Floor project about?
12. What is the Tennis Match Research Dashboard project about?
13. What is the Engineering Team project about?
14. What is the Workout Tracker project about?
15. What are Sergei's top professional skills?
16. How can I contact Sergei or see more of his work online?

In [10]:
faqs_lookup = {faq["faq"]: faq for faq in faqs}

def find_faq(question_number: int):
    faq = faqs_lookup.get(question_number)
    if faq:
        return f"### Question {faq['faq']} is:\n{faq['question']}\n### Sergei's answer:\n{faq['answer']}"
    else:
        return "That question number was not found in the FAQ."

@function_tool
def faq_tool(question_number: int) -> str:
    """Use this tool to retrieve the answer to a frequently asked question by its number."""
    return find_faq(question_number)

In [11]:
EXAMPLES = ["What projects has Sergei built?", "Why did Sergei move into AI engineering?", "What are Sergei's top skills?"]

In [12]:
convo = SQLiteSession("test_conversation")

## New and improved Agentic RAG!

In [13]:
def has_instant_answer(message: str):
    stripped = message.strip().lower()
    return stripped.startswith("q") and stripped[1:].isdigit() and len(stripped) <= 3

def get_instant_answer(message: str):
    question_number = int(message.strip()[1:])
    return find_faq(question_number)

In [14]:
def text_from_event(event):
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        return event.data.delta
    elif event.type == "run_item_stream_event":
        if event.name == "tool_called":
            tool_name = getattr(event.item, "tool_name", None)
            return f'<small class="tool-status">Calling {tool_name}...</small>'
        elif event.name == "tool_output":
            return '<small class="tool-status">Tool returned. Thinking...</small>'

In [15]:
async def run(message):
    async with MCPServerStdio(params=vectorstore_params, client_session_timeout_seconds=120) as vectorstore_mcp:
        cumulative = ""
        agent = Agent(name="Digital Twin", model=model, instructions=instructions, tools=[faq_tool], mcp_servers=[vectorstore_mcp])
        response = Runner.run_streamed(agent, message, session=convo)
        async for event in response.stream_events():
            if (text := text_from_event(event)):
                cumulative += text
                yield cumulative

In [16]:
async def chat(message, history):
    if has_instant_answer(message):
        answer = get_instant_answer(message)
        await convo.add_items([{"role": "user", "content": message}, {"role": "assistant", "content": answer}])
        yield answer
    else:
        async for result in run(message):
            yield result

In [18]:
from styles import CSS, JS
gr.ChatInterface(chat, examples=EXAMPLES, chatbot=gr.Chatbot(show_label=False, height=800)).launch(css=CSS, js=JS, theme=gr.themes.Base(), inbrowser=True)

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


[178536:178536:0913/120628.760238:ERROR:dbus/object_proxy.cc:572] Failed to call method: org.freedesktop.DBus.Properties.GetAll: object_path= /org/freedesktop/UPower/devices/DisplayDevice: org.freedesktop.DBus.Error.ServiceUnknown: The name org.freedesktop.UPower was not provided by any .service files
[178663:9:0913/120628.763156:ERROR:gpu/ipc/client/command_buffer_proxy_impl.cc:285] ContextResult::kTransientFailure: Failed to send GpuControl.CreateCommandBuffer.
Created TensorFlow Lite XNNPACK delegate for CPU.
[178536:178566:0913/120630.857857:ERROR:google_apis/gcm/engine/registration_request.cc:291] Registration response error message: DEPRECATED_ENDPOINT


In [ ]:
gr.close_all()